In [3]:
# 1. 导入必要的包
import os
import sys
import pandas as pd
from datasets import load_from_disk, Dataset, DatasetDict
from tqdm import tqdm

print("✅ 导入成功")

✅ 导入成功


In [4]:
# 2. 设置路径
raw_dataset_path = "/home/y-guo/self-ensemble/new_datasets/my_hotpot_paraphrase"
print(f"原始数据集路径: {raw_dataset_path}")
print(f"路径存在: {os.path.exists(raw_dataset_path)}")

原始数据集路径: /home/y-guo/self-ensemble/new_datasets/my_hotpot_paraphrase
路径存在: True


In [5]:
# 3. 加载原始数据集
print("加载原始数据集...")
raw_ds = load_from_disk(raw_dataset_path)
print(f"原始数据集类型: {type(raw_ds)}")
print(f"是否为 DatasetDict: {isinstance(raw_ds, DatasetDict)}")

if isinstance(raw_ds, DatasetDict):
    print(f"DatasetDict keys: {raw_ds.keys()}")
    raw_ds = raw_ds["train"]
    print("已提取 'train' split")

print(f"\n数据集大小: {len(raw_ds)}")
print(f"列名: {raw_ds.column_names}")

加载原始数据集...
原始数据集类型: <class 'datasets.arrow_dataset.Dataset'>
是否为 DatasetDict: False

数据集大小: 1000
列名: ['uuid', 'answers', 'manual_paraphrases', 'auto_paraphrases']


In [6]:
# 4. 查看第一条样本
print("第一条样本:")
first_sample = raw_ds[0]
for key, value in first_sample.items():
    print(f"  {key}: {value}")

第一条样本:
  uuid: 5a7a06935542990198eaf050
  answers: ["Arthur's Magazine"]
  manual_paraphrases: ["Which magazine was started first Arthur's Magazine or First for Women?"]
  auto_paraphrases: ["Which magazine began publication earlier: Arthur's Magazine or First for Women?", "Between Arthur's Magazine and First for Women, which was founded first?", "Which of the two magazines was launched earlier, Arthur's Magazine or First for Women?", "Which magazine had its start before the other, Arthur's Magazine or First for Women?", "Which came into existence first: Arthur's Magazine or First for Women?", "Which magazine was established earlier, Arthur's Magazine or First for Women?", "Which was published earlier in history, Arthur's Magazine or First for Women?", "Which magazine's first issue preceded the other's: Arthur's Magazine or First for Women?", "Which of these magazines started publication earlier: Arthur's Magazine or First for Women?", "Which magazine has an earlier founding date, Arth

In [7]:
# 5. 转换为 pandas DataFrame
print("转换为 DataFrame...")
df = raw_ds.to_pandas()
print(f"DataFrame shape: {df.shape}")
print(f"\nDataFrame 列:")
print(df.columns.tolist())
print(f"\n前 3 行:")
df.head(3)

转换为 DataFrame...
DataFrame shape: (1000, 4)

DataFrame 列:
['uuid', 'answers', 'manual_paraphrases', 'auto_paraphrases']

前 3 行:


,uuid,answers,manual_paraphrases,auto_paraphrases
0,5a7a06935542990198eaf050,[Arthur's Magazine],[Which magazine was started first Arthur's Mag...,[Which magazine began publication earlier: Art...
1,5a879ab05542996e4f30887e,[Delhi],[The Oberoi family is part of a hotel company ...,[In which city is the head office of the hotel...
2,5a8d7341554299441c6b9fe5,[President Richard Nixon],[Musician and satirist Allie Goertz wrote a so...,[Who was the namesake for The Simpsons charact...


In [8]:
# 6. 查看 orig_id 分组情况
print("orig_id 分组统计:")
grouped = df.groupby("orig_id").size()
print(f"总共 {len(grouped)} 个不同的 orig_id")
print(f"每个 orig_id 的 paraphrase 数量统计:")
print(grouped.value_counts())

orig_id 分组统计:


KeyError: 'orig_id'

In [7]:
# 7. 处理一个样本 orig_id
print("处理第一个 orig_id...")
first_orig_id = df["orig_id"].iloc[0]
print(f"orig_id: {first_orig_id}")

sdf = df[df["orig_id"] == first_orig_id]
sdf = sdf.sort_values("paraphrase_idx")

print(f"\n该 orig_id 有 {len(sdf)} 个 paraphrases:")
for idx, row in sdf.iterrows():
    print(f"  [{row['paraphrase_idx']}] {row['question'][:60]}...")

处理第一个 orig_id...
orig_id: 075e483d21c29a511267ef62bedc0461

该 orig_id 有 11 个 paraphrases:
  [0.0] ...undermine the steps the school had taken to reform?...
  [1.0] ...undo the progress the school had worked toward?...
  [2.0] ...negate the initiatives the school had implemented to impr...
  [3.0] ...set back the reforms the school had been pursuing?...
  [4.0] ...thwart the changes the school had attempted to make?...
  [5.0] ...reverse the improvements the school had tried to accompli...
  [6.0] ...frustrate the measures the school had introduced to turn ...
  [7.0] ...sabotage the transitional actions the school had put in p...
  [8.0] ...devalue the corrective actions the school had undertaken?...
  [9.0] ...cripple the remedial efforts the school had been carrying...
  [nan] The sanctions against the school were a punishing blow, and ...


In [8]:
# 8. 提取 choices 和答案
first = sdf.iloc[0]
labels = first["choices"]["label"]
texts = first["choices"]["text"]
answer_key = first["answerKey"]

print(f"问题: {first['orig_question']}")
print(f"\n选项:")
for label, text in zip(labels, texts):
    marker = "✓" if label == answer_key else " "
    print(f"  {marker} {label}: {text}")
print(f"\n正确答案: {answer_key}")

label2text = {l: t for l, t in zip(labels, texts)}
answer_text = label2text.get(answer_key, "")
print(f"答案文本: {answer_text}")

问题: The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?

选项:
  ✓ A: ignore
    B: enforce
    C: authoritarian
    D: yell at
    E: avoid

正确答案: A
答案文本: ignore


In [9]:
# 9. 聚合所有 orig_id 的数据
print("聚合所有 orig_id...")
items = []
for orig_id, sdf in tqdm(df.groupby("orig_id"), desc="Processing"):
    sdf = sdf.sort_values("paraphrase_idx")
    paraphrases = sdf["question"].tolist()
    first = sdf.iloc[0]
    labels = first["choices"]["label"]
    texts = first["choices"]["text"]
    label2text = {l: t for l, t in zip(labels, texts)}
    answer_key = first["answerKey"]
    answer_text = label2text.get(answer_key, "")
    items.append({
        "uuid": orig_id,
        "paraphrases": paraphrases,
        "answers": [answer_text],
        "answer_label": answer_key,
        "choices_label": labels,
        "choices_text": texts,
        "orig_question": first.get("orig_question", ""),
        "question_concept": first.get("question_concept", ""),
    })

print(f"\n✅ 聚合完成，共 {len(items)} 条")

聚合所有 orig_id...


Processing: 100%|██████████| 50/50 [00:00<00:00, 1154.16it/s]


✅ 聚合完成，共 50 条


In [10]:
# 10. 查看聚合后的第一条数据
print("聚合后第一条数据:")
first_item = items[0]
print(f"uuid: {first_item['uuid']}")
print(f"paraphrases 数量: {len(first_item['paraphrases'])}")
print(f"answers: {first_item['answers']}")
print(f"answer_label: {first_item['answer_label']}")
print(f"\nParaphrases:")
for i, p in enumerate(first_item['paraphrases'][:3]):
    print(f"  [{i}] {p}")
print(f"  ... (共 {len(first_item['paraphrases'])} 个)")

聚合后第一条数据:
uuid: 02e821a3e53cb320790950aab4489e85
paraphrases 数量: 11
answers: ['atlas']
answer_label: D

Paraphrases:
  [0] What have driving navigation apps like Google Maps largely taken the place of?
  [1] Which traditional tool have digital road and highway mapping services mostly supplanted?
  [2] What older resource did highway GPS systems replace for finding routes?
  ... (共 11 个)


In [11]:
# 11. 创建最终的 Dataset
print("创建 HuggingFace Dataset...")
agg_ds = Dataset.from_pandas(pd.DataFrame(items))
print(f"Dataset 大小: {len(agg_ds)}")
print(f"列名: {agg_ds.column_names}")
print(f"\n第一条数据:")
print(agg_ds[0])

创建 HuggingFace Dataset...
Dataset 大小: 50
列名: ['uuid', 'paraphrases', 'answers', 'answer_label', 'choices_label', 'choices_text', 'orig_question', 'question_concept']

第一条数据:
{'uuid': '02e821a3e53cb320790950aab4489e85', 'paraphrases': ['What have driving navigation apps like Google Maps largely taken the place of?', 'Which traditional tool have digital road and highway mapping services mostly supplanted?', 'What older resource did highway GPS systems replace for finding routes?', 'What used to be relied on before modern street navigation apps became common?', 'What conventional item has been overtaken by online driving directions and map services?', 'Before smartphone navigation, what were people using to plan road trips?', 'Which established method of route planning has been made largely obsolete by digital map platforms?', 'What former navigation aid has been replaced by contemporary GPS mapping tools?', 'What prior approach to locating streets and highways has been displaced by map a

In [12]:
# 12. 测试 collate_fn
print("测试 collate_fn...")
batch = [agg_ds[i] for i in range(3)]
print(f"Batch 大小: {len(batch)}")

uuids = [item["uuid"] for item in batch]
answers = [item["answers"] for item in batch]
paraphrases = [item["paraphrases"] for item in batch]

print(f"\nuuids: {uuids}")
print(f"answers: {answers}")
print(f"paraphrases 结构: {len(paraphrases)} items, 每个有 {len(paraphrases[0])} 个 paraphrase")

# 转置 paraphrases
all_paraphrases = list(zip(*paraphrases))
print(f"\n转置后: {len(all_paraphrases)} 个 paraphrase versions, 每个有 {len(all_paraphrases[0])} 个 batch items")

print(f"\n第一个 paraphrase version (所有 batch items):")
for i, p in enumerate(all_paraphrases[0]):
    print(f"  [{i}] {p[:60]}...")

测试 collate_fn...
Batch 大小: 3

uuids: ['02e821a3e53cb320790950aab4489e85', '0476192858e7f541611b5c2d3c5e5197', '075e483d21c29a511267ef62bedc0461']
answers: [['atlas'], ['being found out'], ['ignore']]
paraphrases 结构: 3 items, 每个有 11 个 paraphrase

转置后: 11 个 paraphrase versions, 每个有 3 个 batch items

第一个 paraphrase version (所有 batch items):
  [0] What have driving navigation apps like Google Maps largely t...
  [1] Sean had lied about the corpse and felt terrified; what did ...
  [2] ...undermine the steps the school had taken to reform?...


In [13]:
# 13. 完整测试：使用 Dataset 类
print("\n" + "="*70)
print("完整测试：导入并实例化 CommonsenseParaphraseDataset")
print("="*70)

sys.path.insert(0, '/home/y-guo/self-ensemble/GYB_self-ensemble')
from dataset import CommonsenseParaphraseDataset

print("\n创建 dataset 实例...")
dataset = CommonsenseParaphraseDataset(model_name="llama3.2_3b_it")
print(f"✅ Dataset 创建成功")
print(f"Dataset 大小: {len(dataset.ds)}")
print(f"Dataset root: {dataset.dataset_root}")


完整测试：导入并实例化 CommonsenseParaphraseDataset


/home/y-guo/miniconda3/envs/flexattention/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(



创建 dataset 实例...
Dataset already exists at /home/y-guo/self-ensemble/commonsense_paraphrase/llama3.2_3b_it/paraphrases_dataset. Loading from disk.
✅ Dataset 创建成功
Dataset 大小: 50
Dataset root: /home/y-guo/self-ensemble/commonsense_paraphrase/llama3.2_3b_it


In [14]:
# 14. 测试 dataloader
print("创建 DataLoader...")
dataloader = dataset.get_dataloader(batch_size=2, shuffle=False)
print(f"✅ DataLoader 创建成功")

print("\n获取第一个 batch...")
for uuids, answers, all_paraphrases in dataloader:
    print(f"uuids: {uuids}")
    print(f"answers: {answers}")
    print(f"all_paraphrases 数量: {len(all_paraphrases)} paraphrase versions")
    print(f"每个 version 的 batch size: {len(all_paraphrases[0])}")
    print(f"\n第一个 paraphrase version:")
    for i, p in enumerate(all_paraphrases[0]):
        print(f"  [{i}] {p[:80]}...")
    break

print("\n✅ 所有测试通过！")

创建 DataLoader...
✅ DataLoader 创建成功

获取第一个 batch...
uuids: ['02e821a3e53cb320790950aab4489e85', '0476192858e7f541611b5c2d3c5e5197']
answers: [['atlas'], ['being found out']]
all_paraphrases 数量: 11 paraphrase versions
每个 version 的 batch size: 2

第一个 paraphrase version:
  [0] What have driving navigation apps like Google Maps largely taken the place of?...
  [1] Sean had lied about the corpse and felt terrified; what did he keep worrying abo...

✅ 所有测试通过！


In [19]:
uuids, answers, all_paraphrases = next(iter(dataloader))

In [16]:
uuids

['02e821a3e53cb320790950aab4489e85', '0476192858e7f541611b5c2d3c5e5197']

In [17]:
answers

[['atlas'], ['being found out']]

In [18]:
all_paraphrases = next(iter(dataloader))

In [20]:
all_paraphrases

[('What have driving navigation apps like Google Maps largely taken the place of?',
  'Sean had lied about the corpse and felt terrified; what did he keep worrying about?'),
 ('Which traditional tool have digital road and highway mapping services mostly supplanted?',
  "Having been untruthful about the body and filled with fear, what was on Sean's mind constantly?"),
 ('What older resource did highway GPS systems replace for finding routes?',
  'Sean had been dishonest regarding the body and was very frightened—what did he repeatedly fear?'),
 ('What used to be relied on before modern street navigation apps became common?',
  'After lying about the body and becoming very scared, what was Sean continually anxious about?'),
 ('What conventional item has been overtaken by online driving directions and map services?',
  "Sean wasn't truthful about the body and was extremely afraid; what did his persistent worry concern?"),
 ('Before smartphone navigation, what were people using to plan roa